In [4]:
!pip install pyspark==3.5.0 geopy

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark import SparkContext

# Инициализация Spark
spark = SparkSession.builder.master("local[*]").appName("Colab_Spark_Lab").getOrCreate()
sc = spark.sparkContext

print("Spark запущен! Версия:", spark.version)

Spark запущен! Версия: 3.5.0


## Найти велосипед с максимальным временем пробега

In [ ]:
trips = spark.read.format('csv').option('header', 'true').load("trip.csv")


duration_stats = (
    trips
    .groupby("bike_id")
    .agg(
        F.sum(F.col("duration").cast("int")).alias("duration")
    )
)

# Выборка самого длительного заезда через сортировку
result_limit = duration_stats.sort(F.desc("duration"))
result_limit.limit(1).show()

+-------+--------+
|bike_id|duration|
+-------+--------+
|    593|  974122|
+-------+--------+



## Найти наибольшее геодезическое расстояние между станциями

In [ ]:
stations = spark.read.format('csv').option('header', 'true').load("station.csv")

coords_info = stations.select("id", "lat", "long")

pairs_df = (
    coords_info.selectExpr("id as start_id", "lat as start_lat", "long as start_long")
    .crossJoin(coords_info.selectExpr("id as end_id", "lat as end_lat", "long as end_long"))
)


unique_pairs = pairs_df.filter(F.col("start_id") < F.col("end_id"))


def calculate_gap(lat1, lon1, lat2, lon2):
    try:
        return geodesic((lat1, lon1), (lat2, lon2)).kilometers
    except:
        return 0.0


calculated_distances = unique_pairs.rdd.map(
    lambda p: (
        p.start_id,
        p.end_id,
        calculate_gap(float(p.start_lat), float(p.start_long), float(p.end_lat), float(p.end_long))
    )
)

max_distance_record = calculated_distances.max(key=lambda item: item[2])
max_distance_record

('16', '60', 69.92096757764355)

## Найти путь велосипеда с максимальным временем пробега через станции

In [ ]:

longest_bike = result_limit.first()['bike_id']

bike_route_data = (
    trips.select("id", "bike_id", "start_station_id", "end_station_id", "start_date")
    .where(F.col("bike_id") == longest_bike)

    .withColumn("parsed_date", F.to_timestamp("start_date", "M/d/yyyy H:mm"))
    .sort(F.col("parsed_date"))
    .drop("parsed_date") 
)

bike_route_data.show(bike_route_data.count())

+------+-------+----------------+--------------+----------------+
|    id|bike_id|start_station_id|end_station_id|      start_date|
+------+-------+----------------+--------------+----------------+
|  4296|    593|              47|            64| 8/29/2013 12:00|
|  4426|    593|              64|            64| 8/29/2013 12:52|
|  6524|    593|              70|            72|  8/31/2013 9:07|
|  7184|    593|              66|            39| 8/31/2013 16:11|
|  7315|    593|              39|            73| 8/31/2013 17:41|
|  7327|    593|              73|            47| 8/31/2013 17:57|
|  7378|    593|              47|            66| 8/31/2013 18:50|
|  7684|    593|              66|            76|  9/1/2013 11:57|
|  8350|    593|              50|            65|  9/1/2013 18:20|
|  8710|    593|              65|            68|  9/2/2013 11:31|
|  9839|    593|              68|            74|  9/3/2013 14:49|
|  9876|    593|              74|            76|  9/3/2013 15:27|
| 10665|  

## Найти количество велосипедов в системе

In [ ]:
total_bikes_quantity = trips.select("bike_id").distinct().count()

total_bikes_quantity

698

## Найти пользователей потративших на поездки более 3 часов

In [ ]:

user_activity_stats = (
    trips
    .groupby("zip_code")
    .agg(
        F.sum(F.col("duration").cast("int")).alias("total_duration_sec")
    )
    .where(F.col("total_duration_sec") >= 10800)
    .sort(F.desc("total_duration_sec"))
)


user_activity_stats.show()

+--------+------------------+
|zip_code|total_duration_sec|
+--------+------------------+
|    NULL|          25457657|
|   94107|          11717217|
|   94105|           8698899|
|   94102|           6559224|
|   94133|           5531832|
|   94103|           4507035|
|   94111|           3798169|
|   94109|           3797122|
|   95112|           3733118|
|     nil|           2662507|
|   94117|           2644530|
|   94301|           2469827|
|   94110|           2349119|
|   94041|           2123347|
|   94040|           2075506|
|   94158|           1932820|
|   94108|           1633717|
|   94611|           1414431|
|   94025|           1339944|
|   95113|           1298290|
+--------+------------------+
only showing top 20 rows

